In [1021]:
import pandas as pd

# Define file paths (update these paths with your local file locations)
train_file_path = "/Users/mac-devender/Downloads/dom_project/train.csv"
class_file_path = "/Users/mac-devender/Downloads/dom_project/class.csv"
items_file_path = "/Users/mac-devender/Downloads/dom_project/items.csv"

# Load datasets using pandas
train_data = pd.read_csv(train_file_path, sep='|', encoding='ascii')
class_data = pd.read_csv(class_file_path, sep='|', encoding='ascii')
items_data = pd.read_csv(items_file_path, sep='|', encoding='ascii')

# Display the first few rows of each dataset
print("Train Data:")
print(train_data.head())

print("\nClass Data:")
print(class_data.head())

print("\nItems Data:")
print(items_data.head())


Train Data:
   lineID  day    pid  adFlag  availability  competitorPrice  click  basket  \
0       1    1   6570       0             2            14.60      1       0   
1       2    1  14922       1             1             8.57      0       1   
2       3    1  16382       0             1            14.77      0       1   
3       4    1   1145       1             1             6.59      0       0   
4       5    1   3394       0             1             4.39      0       0   

   order  price  revenue  
0      0  16.89     0.00  
1      0   8.75     0.00  
2      0  16.06     0.00  
3      1   6.55     6.55  
4      1   4.14     4.14  

Class Data:
   lineID  day    pid  adFlag  availability  competitorPrice  price
0       1   93   4772       0             1            11.54  12.04
1       2   93  11548       0             2             6.84   8.60
2       3   93   1958       0             1             9.67  10.39
3       4   93  15071       0             2            17.37  16.4

check missing values

In [1023]:
# Check for missing values
train_missing = train_data.isnull().sum()
items_missing = items_data.isnull().sum()
class_missing = class_data.isnull().sum()

print("Train Data Missing Values:")
print(train_missing)
print("\nItems Data Missing Values:")
print(items_missing)
print("\nClass Data Missing Values:")
print(class_missing)


Train Data Missing Values:
lineID                  0
day                     0
pid                     0
adFlag                  0
availability            0
competitorPrice    100687
click                   0
basket                  0
order                   0
price                   0
revenue                 0
dtype: int64

Items Data Missing Values:
pid                   0
manufacturer          0
group                 0
content               0
unit                  0
pharmForm          2327
genericProduct        0
salesIndex            0
category           4627
campaignIndex     20697
rrp                   0
dtype: int64

Class Data Missing Values:
lineID                 0
day                    0
pid                    0
adFlag                 0
availability           0
competitorPrice    38005
price                  0
dtype: int64


check duplicate values

In [1024]:
# Check for duplicate rows
train_duplicates = train_data.duplicated().sum()
items_duplicates = items_data.duplicated().sum()
class_duplicates = class_data.duplicated().sum()

print(f"Train Data Duplicates: {train_duplicates}")
print(f"Items Data Duplicates: {items_duplicates}")
print(f"Class Data Duplicates: {class_duplicates}")


Train Data Duplicates: 0
Items Data Duplicates: 0
Class Data Duplicates: 0


Identifying and Removing outliers

In [1025]:
# Function to find and remove outliers using IQR
def find_and_remove_outliers(df, column_name):
    # Calculate Q1, Q3, and IQR
    Q1 = df[column_name].quantile(0.25)
    Q3 = df[column_name].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identify rows with outliers
    outliers = (df[column_name] < lower_bound) | (df[column_name] > upper_bound)
    
    # Remove the rows with outliers
    df_cleaned = df[~outliers]  # ~ symbol negates the condition, so it keeps rows without outliers
    return df_cleaned

# Apply the function to relevant columns and remove outliers for `train_data`, `class_data`, and `items_data`

# For train data 
train_data_cleaned = find_and_remove_outliers(train_data, 'price')
train_data_cleaned = find_and_remove_outliers(train_data_cleaned, 'competitorPrice')
train_data_cleaned = find_and_remove_outliers(train_data_cleaned, 'availability')
train_data_cleaned = find_and_remove_outliers(train_data_cleaned, 'revenue')

# For class data (classification period)
class_data_cleaned = find_and_remove_outliers(class_data, 'competitorPrice')
class_data_cleaned = find_and_remove_outliers(class_data_cleaned, 'price')
class_data_cleaned = find_and_remove_outliers(class_data_cleaned, 'availability')

# For items data (static attributes)
items_data_cleaned = find_and_remove_outliers(items_data, 'salesIndex')
items_data_cleaned = find_and_remove_outliers(items_data_cleaned, 'rrp')



In [1026]:
skewness = train_data_cleaned['competitorPrice'].skew()
print(f"Skewness: {skewness}")
skewness = class_data_cleaned['competitorPrice'].skew()
print(f"Skewness: {skewness}")

Skewness: 0.9433650398997195
Skewness: 0.8690475661118741


since data is positively skewed, we will handle missing value of competitor price using median

In [1027]:
# Impute missing competitorPrice with the mean or median

train_data_cleaned['competitorPrice'] = train_data_cleaned['competitorPrice'].fillna(train_data_cleaned['competitorPrice'].median()) 
class_data_cleaned['competitorPrice'] = class_data_cleaned['competitorPrice'].fillna(class_data_cleaned['competitorPrice'].median()) 



In [1028]:
merged_train_data = pd.merge(train_data_cleaned, items_data_cleaned, on="pid", how="left")
merged_train_data


,lineID,day,pid,adFlag,availability,competitorPrice,click,basket,order,price,...,manufacturer,group,content,unit,pharmForm,genericProduct,salesIndex,category,campaignIndex,rrp
0,2,1,14922,1,1,8.57,0,1,0,8.75,...,18.0,1COJ0FIK,50,ST,TAB,1.0,40.0,66.0,C,18.81
1,3,1,16382,0,1,14.77,0,1,0,16.06,...,41.0,22OI7,2X50,ML,STI,0.0,53.0,40.0,NaN,18.48
2,4,1,1145,1,1,6.59,0,0,1,6.55,...,52.0,18OZ00IS,60,G,GEL,0.0,40.0,25.0,NaN,9.31
3,5,1,3394,0,1,4.39,0,0,1,4.14,...,90.0,20OI0,25X2,ST,KOM,0.0,53.0,14.0,NaN,8.13
4,7,1,3856,1,1,3.03,0,0,1,3.58,...,84.0,13OK0FOK,20,G,SAL,0.0,40.0,90.0,NaN,5.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1910993,2755995,92,8096,0,1,2.86,1,0,0,3.25,...,89.0,22OI5,150,ML,SPR,0.0,53.0,12.0,NaN,4.32
1910994,2755999,92,15767,0,1,22.41,1,0,0,18.64,...,917.0,22OIE,15,ML,LOT,0.0,53.0,15.0,NaN,24.75
1910995,2756001,92,2944,0,1,4.71,1,0,0,5.59,...,334.0,21OKF,25,ST,DRA,0.0,53.0,1.0,NaN,5.88
1910996,2756002,92,3853,1,1,6.59,0,1,0,6.33,...,84.0,13OK0FOK,50,G,SAL,0.0,40.0,90.0,A,9.58


In [1030]:
# Drop the 'lineID', 'pid' columns as they are not required
# Drop 'manufacturer, 'category', 'pharmForm', 'group', 'content' as they are categorical variables with large number of unique values
merged_train_data = merged_train_data.drop(columns=['lineID','manufacturer','category','pid','pharmForm','group','content'])

Corelated features

In [1031]:
import numpy as np
# Select only numerical columns
numerical_data = merged_train_data.select_dtypes(include=['number', 'float64', 'int64'])

# Compute the correlation matrix
corr_matrix = numerical_data.corr()

# Create an upper triangle of the correlation matrix (to avoid duplicate checks)
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Identify columns to drop (those with correlation > 0.9)
high_corr_pairs = [(column, row) for column in upper_triangle.columns 
                   for row in upper_triangle.index if upper_triangle[column][row] > 0.9]
high_corr_pairs


[('price', 'competitorPrice'),
 ('revenue', 'order'),
 ('rrp', 'competitorPrice'),
 ('rrp', 'price')]

In [1032]:
# Drop the 'competitorPrice', 'rrp' columns due to corelation > 0.9
merged_train_data = merged_train_data.drop(columns=['competitorPrice','rrp'])


In [1033]:
# Replace NaN with a new category
merged_train_data['unit'] = merged_train_data['unit'].astype(str).replace('nan', 'Missing')
merged_train_data['campaignIndex'] = merged_train_data['campaignIndex'].astype(str).replace('nan', 'Missing')


In [1034]:
# Convert 'salesIndex', 'genericProduct' to numeric, forcing errors to NaN
merged_train_data['salesIndex'] = pd.to_numeric(merged_train_data['salesIndex'], errors='coerce')
merged_train_data['genericProduct'] = pd.to_numeric(merged_train_data['genericProduct'], errors='coerce')

# Impute missing values with the median
merged_train_data['salesIndex'] = merged_train_data['salesIndex'].fillna(merged_train_data['salesIndex'].median())
merged_train_data['genericProduct'] = merged_train_data['genericProduct'].fillna(merged_train_data['genericProduct'].median())

In [1035]:
# List of categorical columns 
categorical_columns = ['campaignIndex','unit']

# Apply one-hot encoding using pandas get_dummies
merged_train_data_encoded = pd.get_dummies(merged_train_data, columns=categorical_columns)

In [1036]:
# Randomly select 1000 rows
final_dataset = merged_train_data_encoded.sample(n=1000, random_state=42)  # Use random_state for reproducibility

In [1037]:
from sklearn.model_selection import train_test_split
X = final_dataset.drop(columns=['revenue'])
y = final_dataset['revenue']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
merged_train_data_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1910998 entries, 0 to 1910997
Data columns (total 22 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   day                    int64  
 1   adFlag                 int64  
 2   availability           int64  
 3   click                  int64  
 4   basket                 int64  
 5   order                  int64  
 6   price                  float64
 7   revenue                float64
 8   genericProduct         float64
 9   salesIndex             float64
 10  campaignIndex_A        bool   
 11  campaignIndex_B        bool   
 12  campaignIndex_C        bool   
 13  campaignIndex_Missing  bool   
 14  unit_G                 bool   
 15  unit_KG                bool   
 16  unit_L                 bool   
 17  unit_M                 bool   
 18  unit_ML                bool   
 19  unit_Missing           bool   
 20  unit_P                 bool   
 21  unit_ST                bool   
dtypes: bool(12), float

Linear Regression and Model Evaluation

In [1038]:
from sklearn.linear_model import LinearRegression

# Initialize and train the model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_val)


In [1039]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_val, y_pred_lr)
print(f"Mean Squared Error: {mse}")


Mean Squared Error: 0.5468403915312331


In [1040]:
import numpy as np
rmse = np.sqrt(mse)
print(f"Root Mean Squared Error: {rmse}")

Root Mean Squared Error: 0.7394865729215325


In [1041]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_val, y_pred_lr)
print(f"Mean Absolute Error: {mae}")

Mean Absolute Error: 0.3113673787816216


In [1042]:
from sklearn.metrics import r2_score
r2 = r2_score(y_val, y_pred_lr)
print(f"R² Score: {r2}")

R² Score: 0.7847126604521278


In [1043]:
import numpy as np

# Avoid division by zero
non_zero_y_val = y_val != 0
mape = np.mean(np.abs((y_val[non_zero_y_val] - y_pred_lr[non_zero_y_val]) / y_val[non_zero_y_val])) * 100
print(f"Mean Absolute Percentage Error: {mape}%")


Mean Absolute Percentage Error: 72.43861065318676%


Random Forest Regressor and Model Evaluation

In [1044]:
from sklearn.ensemble import RandomForestRegressor

# Initialize and train the model
rf_model = RandomForestRegressor(n_estimators=10, random_state=42)  # Use fewer trees
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_val)

In [1045]:
from sklearn.metrics import mean_squared_error
mse_rf = mean_squared_error(y_val, y_pred_rf)
print(f"Mean Squared Error: {mse_rf}")

Mean Squared Error: 0.22876552500000003


In [1046]:
import numpy as np
rmse_rf = np.sqrt(mse_rf)
print(f"Root Mean Squared Error: {rmse_rf}")

Root Mean Squared Error: 0.4782943915623515


In [1047]:
from sklearn.metrics import mean_absolute_error
mae_rf = mean_absolute_error(y_val, y_pred_rf)
print(f"Mean Absolute Error: {mae_rf}")

Mean Absolute Error: 0.09533500000000002


In [1048]:
from sklearn.metrics import r2_score
r2_rf = r2_score(y_val, y_pred_rf)
print(f"R² Score: {r2_rf}")

R² Score: 0.909936570121285


In [1049]:
import numpy as np

# Avoid division by zero
non_zero_y_val = y_val != 0
mape_rf = np.mean(np.abs((y_val[non_zero_y_val] - y_pred_rf[non_zero_y_val]) / y_val[non_zero_y_val])) * 100
print(f"Mean Absolute Percentage Error: {mape_rf}%")

Mean Absolute Percentage Error: 27.509819382052054%


Decision Tree and Model Evaluation

In [1050]:
from sklearn.tree import DecisionTreeRegressor

# Initialize the model
dt_model = DecisionTreeRegressor(random_state=42)

# Fit the model on the reduced training data
dt_model.fit(X_train, y_train)

# Make predictions on the validation set
y_pred_dt = dt_model.predict(X_val)

In [1051]:
from sklearn.metrics import mean_squared_error
mse_dt = mean_squared_error(y_val, y_pred_dt)
print(f"Mean Squared Error: {mse_dt}")

Mean Squared Error: 0.493341


In [1052]:
import numpy as np
rmse_dt = np.sqrt(mse_dt)
print(f"Root Mean Squared Error: {rmse_dt}")

Root Mean Squared Error: 0.7023823744941212


In [1053]:
from sklearn.metrics import mean_absolute_error
mae_dt = mean_absolute_error(y_val, y_pred_dt)
print(f"Mean Absolute Error: {mae_dt}")

Mean Absolute Error: 0.12040000000000002


In [1054]:
from sklearn.metrics import r2_score
r2_dt = r2_score(y_val, y_pred_dt)
print(f"R² Score: {r2_dt}")

R² Score: 0.8057750067026266


In [1055]:
import numpy as np

# Avoid division by zero
non_zero_y_val = y_val != 0
mape_dt = np.mean(np.abs((y_val[non_zero_y_val] - y_pred_dt[non_zero_y_val]) / y_val[non_zero_y_val])) * 100
print(f"Mean Absolute Percentage Error: {mape_dt}%")

Mean Absolute Percentage Error: 33.33225506102367%


Neural Networks and Model Evaluation

In [1056]:
from sklearn.neural_network import MLPRegressor

mlp_model = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
mlp_model.fit(X_train, y_train)
y_pred_mlp = mlp_model.predict(X_val)

In [1057]:
from sklearn.metrics import mean_squared_error
mse_mlp = mean_squared_error(y_val, y_pred_mlp)
print(f"Mean Squared Error: {mse_mlp}")

Mean Squared Error: 0.2557678005077555


In [1058]:
import numpy as np
rmse_mlp = np.sqrt(mse_mlp)
print(f"Root Mean Squared Error: {rmse_mlp}")

Root Mean Squared Error: 0.505734911300135


In [1059]:
from sklearn.metrics import mean_absolute_error
mae_mlp = mean_absolute_error(y_val, y_pred_mlp)
print(f"Mean Absolute Error: {mae_mlp}")

Mean Absolute Error: 0.2620731639281742


In [1060]:
from sklearn.metrics import r2_score
r2_mlp = r2_score(y_val, y_pred_mlp)
print(f"R² Score: {r2_mlp}")

R² Score: 0.8993059580709839


In [1061]:
import numpy as np

# Avoid division by zero
non_zero_y_val = y_val != 0
mape_mlp = np.mean(np.abs((y_val[non_zero_y_val] - y_pred_mlp[non_zero_y_val]) / y_val[non_zero_y_val])) * 100
print(f"Mean Absolute Percentage Error: {mape_mlp}%")

Mean Absolute Percentage Error: 38.53557341206716%


K Nearest Neighbour and Model Evaluation

In [1062]:
from sklearn.neighbors import KNeighborsRegressor

knn_model = KNeighborsRegressor(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_val)

In [1063]:
from sklearn.metrics import mean_squared_error
mse_knn = mean_squared_error(y_val, y_pred_knn)
print(f"Mean Squared Error: {mse_knn}")

Mean Squared Error: 2.7270320800000003


In [1064]:
import numpy as np
rmse_knn = np.sqrt(mse_knn)
print(f"Root Mean Squared Error: {rmse_knn}")

Root Mean Squared Error: 1.6513727865021879


In [1065]:
from sklearn.metrics import mean_absolute_error
mae_knn = mean_absolute_error(y_val, y_pred_knn)
print(f"Mean Absolute Error: {mae_knn}")

Mean Absolute Error: 0.7552800000000001


In [1066]:
from sklearn.metrics import r2_score
r2_knn = r2_score(y_val, y_pred_knn)
print(f"R² Score: {r2_knn}")

R² Score: -0.0736139657148347


In [1067]:
import numpy as np

# Avoid division by zero
non_zero_y_val = y_val != 0
mape_knn = np.mean(np.abs((y_val[non_zero_y_val] - y_pred_knn[non_zero_y_val]) / y_val[non_zero_y_val])) * 100
print(f"Mean Absolute Percentage Error: {mape_knn}%")

Mean Absolute Percentage Error: 76.87955577415144%
